---
title: Return Statistics
execute:
  enabled: true
---

This page's code cells are **executed live** by Quarto (via Jupyter) every time the
docs are built. `q.stats` computes return-stream statistics with no Plotly
dependency; `q.plot` renders them. See the [Interactive Demo](demo.ipynb) for
feature engineering and price-chart examples.

## Real return streams

We use qrt's bundled sample datasets — AAPL as the "strategy" and SPY as the
benchmark — loaded offline via `q.data.datasets.load`, no network dependency:

In [1]:
import pandas as pd

import qrt as q

aapl = q.data.datasets.load("aapl")
spy = q.data.datasets.load("spy")

returns = pd.concat(
    {
        "AAPL": aapl["close"].pct_change(),
        "SPY": spy["close"].pct_change(),
    },
    axis=1,
).dropna()
strategy = returns["AAPL"]
benchmark = returns["SPY"]

returns.tail()

,AAPL,SPY
datetime,,
2026-07-10,-0.002846,0.004310
2026-07-13,0.006311,-0.007656
2026-07-14,-0.007721,0.003551
2026-07-15,0.040145,0.003964
2026-07-16,0.017588,-0.005419


## Headline performance

`q.stats.performance` returns a pandas `Series` of Total Return, CAGR, Volatility,
Sharpe, Sortino, Calmar, Max Drawdown, Win Rate, and Periods. The annualization
frequency is inferred from the index (252 for this daily series) unless
`periods_per_year` is given explicitly:

In [2]:
q.stats.performance(strategy)

Total Return     396.815928
CAGR               0.253688
Volatility         0.383120
Sharpe             0.789515
Sortino            1.128558
Calmar             0.310126
Max Drawdown      -0.818014
Win Rate           0.523231
Periods         6672.000000
Name: AAPL, dtype: float64

## Benchmark-relative: alpha and beta

`q.stats.beta` measures sensitivity to the benchmark; `q.stats.alpha` is the
annualized Jensen alpha unexplained by that beta exposure. `q.stats.benchmark_stats`
combines both with active return, correlation, tracking error, and information
ratio:

In [3]:
print(f"Beta: {q.stats.beta(strategy, benchmark):.2f}")
print(f"Alpha: {q.stats.alpha(strategy, benchmark):.2%}")

q.stats.benchmark_stats(strategy, benchmark)

Beta: 1.13
Alpha: 19.14%


Strategy Total Return      396.815928
Benchmark Total Return       7.237651
Active Return               47.292400
Beta                         1.129953
Alpha                        0.191428
Correlation                  0.569114
Tracking Error               0.316021
Information Ratio            0.646161
Periods                   6672.000000
Name: AAPL vs SPY, dtype: float64

## Rolling diagnostics

Rolling metrics make regime changes visible. A 63-business-day window is roughly
one quarter. The four diagnostics are combined into one interactive figure with
`q.plot.col`:

In [4]:
window = 63
rolling = pd.concat(
    {
        "Volatility": q.stats.rolling_volatility(strategy, window),
        "Sharpe": q.stats.rolling_sharpe(strategy, window),
        "Beta": q.stats.rolling_beta(strategy, benchmark, window),
        "Alpha": q.stats.rolling_alpha(strategy, benchmark, window),
    },
    axis=1,
)

fig = q.plot.col(
    rolling,
    title=f"{window}-day rolling risk and benchmark diagnostics",
    ylabel="Value",
    height=550,
)
fig.show()

## Calendar returns

`q.stats.monthly_returns` compounds daily returns within each calendar month into a
year-by-month table; `q.plot.monthly_heatmap` renders it as an interactive heatmap:

In [5]:
display(q.stats.monthly_returns(strategy).style.format("{:.1%}"))

fig = q.plot.monthly_heatmap(strategy, title="Strategy monthly returns")
fig.show()

Month,1,2,3,4,5,6,7,8,9,10,11,12
Year,,,,,,,,,,,,
2000,-7.3%,10.5%,18.5%,-8.7%,-32.3%,24.7%,-3.0%,19.9%,-57.7%,-24.0%,-15.7%,-9.8%
2001,45.4%,-15.6%,20.9%,15.5%,-21.7%,16.5%,-19.2%,-1.3%,-16.4%,13.2%,21.3%,2.8%
2002,12.9%,-12.2%,9.1%,2.5%,-4.0%,-23.9%,-13.9%,-3.3%,-1.7%,10.8%,-3.5%,-7.5%
2003,0.2%,4.5%,-5.8%,0.6%,26.2%,6.2%,10.6%,7.3%,-8.4%,10.5%,-8.7%,2.2%
2004,5.6%,6.0%,13.0%,-4.7%,8.8%,16.0%,-0.6%,6.6%,12.4%,35.2%,28.0%,-4.0%
2005,19.4%,16.7%,-7.1%,-13.5%,10.3%,-7.4%,15.9%,9.9%,14.3%,7.4%,17.8%,6.0%
2006,5.0%,-9.3%,-8.4%,12.2%,-15.1%,-4.2%,18.7%,-0.2%,13.5%,5.3%,13.0%,-7.4%
2007,1.0%,-1.3%,9.8%,7.4%,21.4%,0.7%,8.0%,5.1%,10.8%,23.8%,-4.1%,8.7%
2008,-31.7%,-7.6%,14.8%,21.2%,8.5%,-11.3%,-5.1%,6.7%,-33.0%,-5.3%,-13.9%,-7.9%


## Chained API: `q.stats.returns(...)`

For notebook exploration, `q.stats.returns()` binds a return stream (and optional
benchmark) once and exposes the same stats as methods, plus `.plot(kind=...)` which
delegates to `q.plot`. Each call creates a fresh, independent object — there is no
hidden global state to reason about:

In [6]:
bound = q.stats.returns(strategy, benchmark=benchmark)

bound.performance()

Total Return     396.815928
CAGR               0.253688
Volatility         0.383120
Sharpe             0.789515
Sortino            1.128558
Calmar             0.310126
Max Drawdown      -0.818014
Win Rate           0.523231
Periods         6672.000000
Name: AAPL, dtype: float64

`kind` accepts `"performance"`/`"tearsheet"` (equity + drawdown report), `"equity"`,
`"drawdown"`, or `"monthly_heatmap"`. The bound benchmark is passed through
automatically for the report variants:

In [7]:
print(f"Beta: {bound.beta():.2f}")
print(f"Alpha: {bound.alpha():.2%}")

fig = bound.plot("performance")
fig.show()

Beta: 1.13
Alpha: 19.14%
